# Data exploration nach CRISP-DM

In [ ]:
#!/usr/bin/env python3

# Bibliotheken laden
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display # Für die Anzeige von DataFrames in Jupyter Notebooks
import os # Für Betriebssystem- und Pfadoperationen
from pathlib import Path # Für Pfadoperationen
from concurrent.futures import ThreadPoolExecutor # Für parallele Verarbeitung
from tqdm import tqdm # Für Fortschrittsbalken
from tqdm.notebook import tqdm # Für Fortschrittsbalken in Jupyter Notebooks


# Pfade definieren
price_path = os.path.expanduser("~/PredAna_Python/tankerkoenig-data-working/prices/")
price_path_example = price_path + "2024/04/2024-04-01-prices.csv"
station_path = os.path.expanduser("~/PredAna_Python/tankerkoenig-data-working/stations/")


# Daten laden
df_prices = pd.read_csv(price_path_example)
df_stations = pd.read_csv(station_path + "stations.csv")

# Daten anzeigen
print("Übersicht Preise:\n")
display(df_prices.head())
print("\n")  # Zeilenumbruch für bessere Lesbarkeit
print("Übersicht Stationen:\n")
display(df_stations.head())

# Übersicht über die Daten
print("Statistiken über df_prices:\n")
display(df_prices.describe())
print("\n")  # Zeilenumbruch für bessere Lesbarkeit
print("Statistiken über df_stations:\n")
display(df_stations.describe())

# Fehlende Werte prüfen
print("Fehlende Werte in df_prices:\n" + str(df_prices.isnull().sum()))
print("\n")  # Zeilenumbruch für bessere Lesbarkeit
print("Fehlende Werte in df_stations:\n" + str(df_stations.isnull().sum()))


# rekursiv iterieren über den Pfad price_path, um Fehlende Werte zu analysieren und aufzusummieren
PRICE_COLS = ["diesel", "e5", "e10"]
CHANGE_COLS = ["dieselchange", "e5change", "e10change"]
USECOLS = PRICE_COLS + CHANGE_COLS

DTYPES = {
    "diesel": "float32",
    "e5": "float32",
    "e10": "float32",
    "dieselchange": "int8",
    "e5change": "int8",
    "e10change": "int8",
}

def analyze_price_file(filepath: str):
    df = pd.read_csv(filepath, usecols=USECOLS, dtype=DTYPES)

    nan_count = int(df[PRICE_COLS].isna().sum().sum())
    zero_count = int((df[PRICE_COLS] == 0).sum().sum())

    zero_removed = int(
        ((df["diesel"] == 0) & (df["dieselchange"] == 2)).sum()
        + ((df["e5"] == 0) & (df["e5change"] == 2)).sum()
        + ((df["e10"] == 0) & (df["e10change"] == 2)).sum()
    )
    zero_not_removed = zero_count - zero_removed
    row_count = len(df)

    return row_count, nan_count, zero_count, zero_removed, zero_not_removed


price_path = Path("~/PredAna_Python/tankerkoenig-data-working/prices/").expanduser()
all_files = [str(p) for p in price_path.rglob("*.csv")]
print(f"Gefundene CSV-Dateien: {len(all_files)}")

total_rows = 0
total_nan = 0
total_zero = 0
total_zero_removed = 0
total_zero_not_removed = 0

with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(tqdm(
        executor.map(analyze_price_file, all_files),
        total=len(all_files),
        desc="Analysiere Preisdateien"
    ))

for row_count, nan_count, zero_count, zero_removed, zero_not_removed in results:
    total_rows += row_count
    total_nan += nan_count
    total_zero += zero_count
    total_zero_removed += zero_removed
    total_zero_not_removed += zero_not_removed

print(f"\nGesamtzahl Zeilen:                  {total_rows}")
print(f"Echte NaN in Preisfeldern:          {total_nan}")
print(f"Preis == 0 in Preisfeldern:         {total_zero}")
print(f"Preis == 0 und change == 2:         {total_zero_removed}")
print(f"Preis == 0 und change != 2:         {total_zero_not_removed}")




# Datentypen prüfen
print("Datentypen in df_prices:\n" + str(df_prices.dtypes))
print("\n")  # Zeilenumbruch für bessere Lesbarkeit
print("Datentypen in df_stations:\n" + str(df_stations.dtypes))





Übersicht Preise:



,date,station_uuid,diesel,e5,e10,dieselchange,e5change,e10change
0,2024-04-01 00:00:34+02,db307731-f6c4-4c45-9af4-1058e9b23397,1.709,1.839,1.779,1,1,1
1,2024-04-01 00:00:34+02,6bb23c12-f7ce-41a8-ad1c-b50854808fe9,1.699,1.779,1.719,1,1,1
2,2024-04-01 00:00:34+02,c7f687c8-5718-49d0-88b0-092c55aeb9a4,1.689,1.879,1.819,1,1,1
3,2024-04-01 00:00:34+02,c1a1a169-d0b9-4e6a-a9a0-a44c4065e8f7,1.829,1.959,1.899,1,1,1
4,2024-04-01 00:00:34+02,a9994385-741f-4374-b738-d33681e101a7,1.699,1.879,1.819,1,0,0




Übersicht Stationen:



,uuid,name,brand,street,house_number,post_code,city,latitude,longitude
0,00060723-0001-4444-8888-acdc00000001,BAGeno Raiffeisen eG,NaN,Künzelsauer Strasse,7,74653,Ingelfingen,49.296822,9.661385
1,005056ba-7cb6-1ed2-bceb-5332ab168d12,famila Tankstelle,FAMILA,Pascalstrasse,9,25442,Quickborn,53.742150,9.941240
2,005056ba-7cb6-1ed2-bceb-573c18314d16,star Tankstelle,STAR,Riehler Strasse,240,50735,Köln,50.961800,6.980070
3,005056ba-7cb6-1ed2-bceb-662ba1a94d1f,star Tankstelle,STAR,BAB 10 / Seeberg Ost,NaN,15345,Altlandsberg,52.550160,13.682120
4,005056ba-7cb6-1ed2-bceb-6f7b23564d23,star Tankstelle,STAR,Duisburger Straße,130,47166,Duisburg,51.489790,6.783730


Statistiken über df_prices:



...



Statistiken über df_stations:



,latitude,longitude
count,15442.000000,15442.000000
mean,50.832523,9.596294
std,2.013957,2.021447
min,0.000000,0.000000
25%,49.385821,8.026218
50%,50.974630,9.297347
75%,52.165426,11.066987
max,55.015890,14.983898


Fehlende Werte in df_prices:
date            0
station_uuid    0
diesel          0
e5              0
e10             0
dieselchange    0
e5change        0
e10change       0
dtype: int64


Fehlende Werte in df_stations:
uuid               0
name               0
brand            544
street             2
house_number    3678
post_code          2
city               3
latitude           0
longitude          0
dtype: int64
Gefundene CSV-Dateien: 4365


Analysiere Preisdateien:   0%|          | 0/4365 [00:00<?, ?it/s]


Gesamtzahl Zeilen:                  1128245397
Echte NaN in Preisfeldern:          0
Preis == 0 in Preisfeldern:         61521119
Preis == 0 und change == 2:         831021
Preis == 0 und change != 2:         60690098
Datentypen in df_prices:
date             object
station_uuid     object
diesel          float64
e5              float64
e10             float64
dieselchange      int64
e5change          int64
e10change         int64
dtype: object


Datentypen in df_stations:
uuid             object
name             object
brand            object
street           object
house_number     object
post_code        object
city             object
latitude        float64
longitude       float64
dtype: object


In [11]:
# Suche nach e5 = 4.0 oder ähnlich hohe werte, die auf Fehler hindeuten könnten

TARGET = 4.0

def find_e5_outlier(filepath: str):
    df = pd.read_csv(
        filepath,
        usecols=["date", "station_uuid", "e5", "e5change"],
        dtype={
            "date": "string",
            "station_uuid": "string",
            "e5": "float32",
            "e5change": "int8",
        }
    )

    mask = np.isclose(df["e5"], TARGET, atol=1e-6)

    count = int(mask.sum())

    if count == 0:
        return None

    hits = df.loc[mask, ["date", "station_uuid", "e5", "e5change"]].copy()
    hits["file"] = filepath

    return {
        "file": filepath,
        "count": count,
        "rows": hits
    }

price_path = Path("~/PredAna_Python/tankerkoenig-data-working/prices/").expanduser()
all_files = [str(p) for p in price_path.rglob("*.csv")]

print(f"Gefundene CSV-Dateien: {len(all_files)}")

results = []

with ThreadPoolExecutor(max_workers=8) as executor:
    mapped = executor.map(find_e5_outlier, all_files)
    for result in tqdm(mapped, total=len(all_files), desc="Suche nach e5 = 4.0"):
        if result is not None:
            results.append(result)

if not results:
    print("Kein Treffer für e5 = 4.0 gefunden.")
else:
    total_hits = sum(r["count"] for r in results)
    files_with_hits = len(results)

    print(f"\nGesamtzahl Treffer für e5 = 4.0: {total_hits}")
    print(f"Anzahl betroffener Dateien: {files_with_hits}")

    summary_df = pd.DataFrame(
        [{"file": r["file"], "count": r["count"]} for r in results]
    ).sort_values("count", ascending=False)

    print("\nTreffer pro Datei:")
    display(summary_df)

    detail_df = pd.concat([r["rows"] for r in results], ignore_index=True)

    print("\nDetailzeilen:")
    display(detail_df.sort_values(["file", "date"]))

Gefundene CSV-Dateien: 4365


Suche nach e5 = 4.0:   0%|          | 0/4365 [00:00<?, ?it/s]


Gesamtzahl Treffer für e5 = 4.0: 21976
Anzahl betroffener Dateien: 833

Treffer pro Datei:


,file,count
321,/Users/danielfeil/PredAna_Python/tankerkoenig-...,56
318,/Users/danielfeil/PredAna_Python/tankerkoenig-...,55
334,/Users/danielfeil/PredAna_Python/tankerkoenig-...,52
285,/Users/danielfeil/PredAna_Python/tankerkoenig-...,51
342,/Users/danielfeil/PredAna_Python/tankerkoenig-...,51
...,...,...
0,/Users/danielfeil/PredAna_Python/tankerkoenig-...,1
619,/Users/danielfeil/PredAna_Python/tankerkoenig-...,1
1,/Users/danielfeil/PredAna_Python/tankerkoenig-...,1
122,/Users/danielfeil/PredAna_Python/tankerkoenig-...,1



Detailzeilen:


,date,station_uuid,e5,e5change,file
21975,2015-08-04 09:46:01+02,8a2cc672-2c7f-4322-a74f-c80336e49c9e,4.0,1,/Users/danielfeil/PredAna_Python/tankerkoenig-...
21974,2015-09-17 12:34:01+02,5b948d66-fcd6-42b9-af3a-468b97ebdc1f,4.0,1,/Users/danielfeil/PredAna_Python/tankerkoenig-...
0,2022-05-04 10:55:09+02,622ccb6f-d4b8-4ed1-bbe4-c1d8a4ad0e4f,4.0,3,/Users/danielfeil/PredAna_Python/tankerkoenig-...
1,2022-07-22 17:04:09+02,e972b837-abef-4cb3-ad42-65c9ccf1f72c,4.0,1,/Users/danielfeil/PredAna_Python/tankerkoenig-...
17159,2023-03-29 18:27:10+02,b9b395af-04c3-4150-a81b-f4dd215a20f0,4.0,3,/Users/danielfeil/PredAna_Python/tankerkoenig-...
...,...,...,...,...,...
4646,2025-09-08 20:50:58+02,78dfc1c1-8e95-4ed2-bad0-14ecdb1e8163,4.0,0,/Users/danielfeil/PredAna_Python/tankerkoenig-...
4647,2025-09-08 21:36:40+02,78dfc1c1-8e95-4ed2-bad0-14ecdb1e8163,4.0,0,/Users/danielfeil/PredAna_Python/tankerkoenig-...
4648,2025-09-08 21:55:55+02,78dfc1c1-8e95-4ed2-bad0-14ecdb1e8163,4.0,0,/Users/danielfeil/PredAna_Python/tankerkoenig-...
4649,2025-09-08 23:33:20+02,78dfc1c1-8e95-4ed2-bad0-14ecdb1e8163,4.0,0,/Users/danielfeil/PredAna_Python/tankerkoenig-...


In [16]:
type(results) 
# len(results)
print("\n".join(summary_df["file"].astype(str).tolist()))
for _, row in summary_df.iterrows(): print(f'{row["count"]:>6}  {row["file"]}')
pd.DataFrame(
    [(row["count"], Path(row["file"]).name) for _, row in summary_df.iterrows()],
    columns=["count", "filename"]
).to_csv("e5_4_0_count_per_file.csv", index=False)

/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-17-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-06-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-13-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/04/2024-04-26-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-08-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-16-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-15-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-20-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-02-prices.csv
/Users/danielfeil/PredAna_Python/tankerkoenig-data-working/prices/2024/05/2024-05-03-prices.csv
/Users/danielfeil/PredAna_Python/tankerk

In [18]:
# verdächtige UUIDs extrahieren

suspect_df = (
    detail_df.groupby("station_uuid")
    .size()
    .reset_index(name="count_e5_eq_4")
    .sort_values("count_e5_eq_4", ascending=False)
)

print(f"Anzahl eindeutiger verdächtiger UUIDs: {len(suspect_df)}")
display(suspect_df.head(20))


Anzahl eindeutiger verdächtiger UUIDs: 17


,station_uuid,count_e5_eq_4
6,78dfc1c1-8e95-4ed2-bad0-14ecdb1e8163,17110
11,b9b395af-04c3-4150-a81b-f4dd215a20f0,3044
10,8e9ace8c-bd06-58a1-9ddc-e480e3b72b57,1339
3,4be6cac8-e125-41eb-ac4f-4103f8735107,466
15,df7ab2a6-e897-44d5-8609-06d35592e7d6,2
14,d9879018-c48a-415e-a944-0c67c9618b36,2
12,c1169472-dc4d-583d-81de-60440247d683,2
9,8be7e249-180e-407a-8f3b-dcfae36c139c,2
0,00099999-99e7-4444-8888-acdc00000076,1
13,d4105647-4b00-4f79-9ae8-065e4cb75218,1


In [20]:
# Liste aller stations UUIDs

stations_path = Path("~/PredAna_Python/tankerkoenig-data-working/stations/").expanduser()
station_files = list(stations_path.rglob("*.csv"))

print(f"Gefundene Stationsdateien: {len(station_files)}")

def read_station_uuids(filepath):
    df = pd.read_csv(
        filepath,
        usecols=["uuid"],
        dtype={"uuid": "string"}
    )
    return set(df["uuid"].dropna().unique())

known_station_ids = set()

with ThreadPoolExecutor(max_workers=8) as executor:
    for uuid_set in tqdm(
        executor.map(read_station_uuids, station_files),
        total=len(station_files),
        desc="Lese Stations-UUIDs"
    ):
        known_station_ids.update(uuid_set)

print(f"Anzahl eindeutiger UUIDs in allen stations.csv: {len(known_station_ids)}")


Gefundene Stationsdateien: 2675


Lese Stations-UUIDs:   0%|          | 0/2675 [00:00<?, ?it/s]

Anzahl eindeutiger UUIDs in allen stations.csv: 17802


In [21]:
# 3) Vergleich
suspect_df["known_in_any_stations_csv"] = suspect_df["station_uuid"].isin(known_station_ids)

# 4) Orphans isolieren
orphan_df = suspect_df.loc[~suspect_df["known_in_any_stations_csv"]].copy()

print(f"Verdächtige UUIDs gesamt:                    {len(suspect_df)}")
print(f"Davon in stations.csv bekannt:               {suspect_df['known_in_any_stations_csv'].sum()}")
print(f"Davon NICHT in stations.csv (Orphans):       {len(orphan_df)}")

display(suspect_df)
display(orphan_df)

# 5) Export
suspect_df.to_csv("e5_4_station_uuid_check.csv", index=False)
orphan_df.to_csv("e5_4_station_uuid_orphans.csv", index=False)

Verdächtige UUIDs gesamt:                    17
Davon in stations.csv bekannt:               17
Davon NICHT in stations.csv (Orphans):       0


,station_uuid,count_e5_eq_4,known_in_any_stations_csv
6,78dfc1c1-8e95-4ed2-bad0-14ecdb1e8163,17110,True
11,b9b395af-04c3-4150-a81b-f4dd215a20f0,3044,True
10,8e9ace8c-bd06-58a1-9ddc-e480e3b72b57,1339,True
3,4be6cac8-e125-41eb-ac4f-4103f8735107,466,True
15,df7ab2a6-e897-44d5-8609-06d35592e7d6,2,True
14,d9879018-c48a-415e-a944-0c67c9618b36,2,True
12,c1169472-dc4d-583d-81de-60440247d683,2,True
9,8be7e249-180e-407a-8f3b-dcfae36c139c,2,True
0,00099999-99e7-4444-8888-acdc00000076,1,True
13,d4105647-4b00-4f79-9ae8-065e4cb75218,1,True


,station_uuid,count_e5_eq_4,known_in_any_stations_csv
